# SAM3.1 Single Image Prompt Test

This notebook tests local SAM3.1 on one camera frame, without video propagation. It runs every text prompt from `configs/sam3_text_prompts_pointwise_v1.yaml`, converts accepted instances to project class ids, and merges them into one semantic mask by max score.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SAM3_ROOT = Path("/Users/makism/git_repo/sam3")
SAM3_MODEL_DIR = SAM3_ROOT / "sam3.1"
SAM3_CHECKPOINT = SAM3_MODEL_DIR / "sam3.1_multiplex.pt"

PROMPT_CONFIG = PROJECT_ROOT / "configs" / "sam3_text_prompts_pointwise_v1.yaml"
CLASSES_YAML = PROJECT_ROOT / "configs" / "classes_pointwise_v1.yaml"
OUT_DIR = PROJECT_ROOT / "output" / "sam3_single_image_test"

# Set this manually if auto-discovery does not find a frame.
IMAGE_PATH = None

MIN_SCORE = 0.70
USE_FA3 = False
MAX_PROMPTS = None  # e.g. 3 for a tiny smoke test

print("PROJECT_ROOT", PROJECT_ROOT)
print("SAM3_ROOT", SAM3_ROOT)
print("SAM3_CHECKPOINT", SAM3_CHECKPOINT)


In [ ]:
import json
import os
import sys
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

sys.path.insert(0, str(SAM3_ROOT.resolve()))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HOME", str(SAM3_MODEL_DIR.resolve()))

assert SAM3_ROOT.is_dir(), f"SAM3 repo not found: {SAM3_ROOT}"
assert SAM3_CHECKPOINT.is_file(), f"SAM3.1 checkpoint not found: {SAM3_CHECKPOINT}"
assert PROMPT_CONFIG.is_file(), f"Prompt config not found: {PROMPT_CONFIG}"
assert CLASSES_YAML.is_file(), f"Classes YAML not found: {CLASSES_YAML}"

OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def strip_quotes(value: str) -> str:
    value = value.strip()
    if len(value) >= 2 and value[0] in {"'", '"'} and value[-1] == value[0]:
        return value[1:-1]
    return value


def load_prompt_config(path: Path) -> dict[str, list[str]]:
    prompts = {}
    current = None
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].rstrip()
        if not line.strip():
            continue
        indent = len(line) - len(line.lstrip(" "))
        stripped = line.strip()
        if indent == 0 and ":" in stripped:
            key, value = [x.strip() for x in stripped.split(":", 1)]
            current = key
            prompts[current] = []
            if value:
                prompts[current].extend(strip_quotes(x) for x in value.strip("[]").split(",") if x.strip())
            continue
        if indent >= 2 and stripped.startswith("- ") and current is not None:
            prompts[current].append(strip_quotes(stripped[2:]))
    return prompts


def load_classes_yaml(path: Path) -> dict[str, int]:
    classes = {}
    in_semantic = False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].rstrip()
        if not line.strip():
            continue
        if line.strip() == "semantic_classes:":
            in_semantic = True
            continue
        if in_semantic:
            indent = len(line) - len(line.lstrip(" "))
            stripped = line.strip()
            if indent == 0:
                break
            if ":" in stripped:
                name, value = [x.strip() for x in stripped.split(":", 1)]
                classes[name] = int(value)
    return classes


def find_first_image(project_root: Path) -> Path | None:
    patterns = [
        "output/**/img2/**/*.jpg",
        "output/**/img2/**/*.jpeg",
        "output/**/img2/**/*.png",
        "output/**/image_2/**/*.jpg",
        "data/**/*.jpg",
        "data/**/*.png",
    ]
    for pattern in patterns:
        matches = sorted(project_root.glob(pattern))
        if matches:
            return matches[0]
    return None


prompts_by_label = load_prompt_config(PROMPT_CONFIG)
class_to_id = load_classes_yaml(CLASSES_YAML)
label_to_id = {label: class_to_id[label] for label in prompts_by_label if label in class_to_id}

flat_prompts = [(label, prompt) for label, prompts in prompts_by_label.items() for prompt in prompts if label in label_to_id]
if MAX_PROMPTS is not None:
    flat_prompts = flat_prompts[:MAX_PROMPTS]

if IMAGE_PATH is None:
    IMAGE_PATH = find_first_image(PROJECT_ROOT)
else:
    IMAGE_PATH = Path(IMAGE_PATH)

assert IMAGE_PATH is not None and Path(IMAGE_PATH).is_file(), "Set IMAGE_PATH to an existing camera frame"
IMAGE_PATH = Path(IMAGE_PATH)

print(f"Image: {IMAGE_PATH}")
print(f"Prompt count: {len(flat_prompts)}")
print("Labels:", sorted(label_to_id.items(), key=lambda x: x[1]))


In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
width, height = image.size
print("Image size:", width, height)

plt.figure(figsize=(12, 7))
plt.imshow(image)
plt.axis("off");


In [ ]:
from sam3.model_builder import build_sam3_multiplex_video_predictor

print("Building SAM3.1 predictor...")
predictor = build_sam3_multiplex_video_predictor(
    checkpoint_path=str(SAM3_CHECKPOINT),
    use_fa3=USE_FA3,
    async_loading_frames=False,
)

print("Starting single-image SAM3 session...")
response = predictor.handle_request(
    request={
        "type": "start_session",
        "resource_path": str(IMAGE_PATH),
    }
)
session_id = response["session_id"]
session_id


In [ ]:
def to_numpy(value):
    if value is None:
        return None
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "cpu"):
        value = value.cpu()
    return np.asarray(value)


def first_present(mapping, keys):
    if not isinstance(mapping, dict):
        return None
    for key in keys:
        if key in mapping:
            return mapping[key]
    return None


def normalize_masks(value):
    arr = to_numpy(value)
    if arr is None:
        return np.zeros((0, height, width), dtype=bool)
    if arr.ndim == 4 and arr.shape[1] == 1:
        arr = arr[:, 0]
    if arr.ndim == 2:
        arr = arr[None, ...]
    if arr.ndim != 3:
        raise ValueError(f"Expected masks [N,H,W], got {arr.shape}")
    return arr if arr.dtype == np.bool_ else arr > 0


def normalize_scores(value, count):
    if value is None:
        return np.ones((count,), dtype=np.float32)
    arr = to_numpy(value).reshape(-1).astype(np.float32)
    if arr.shape != (count,):
        return np.ones((count,), dtype=np.float32)
    return arr


def normalize_boxes(value, count):
    if value is None:
        return None
    arr = to_numpy(value).astype(np.float32)
    if arr.shape == (count, 4):
        return arr
    return None


def extract_arrays(outputs):
    if isinstance(outputs, dict):
        masks_value = first_present(outputs, ("masks", "pred_masks", "out_binary_masks", "video_res_masks", "mask_logits", "out_mask_logits"))
        scores_value = first_present(outputs, ("scores", "pred_scores", "object_scores", "ious", "obj_scores", "out_probs"))
        boxes_value = first_present(outputs, ("boxes_xyxy", "boxes", "pred_boxes", "out_boxes_xywh"))
    elif isinstance(outputs, (tuple, list)) and len(outputs) >= 2:
        masks_value = outputs[1]
        scores_value = outputs[2] if len(outputs) >= 3 else None
        boxes_value = outputs[3] if len(outputs) >= 4 else None
    else:
        raise ValueError(f"Unsupported output type: {type(outputs)!r}")

    masks = normalize_masks(masks_value)
    scores = normalize_scores(scores_value, masks.shape[0])
    boxes = normalize_boxes(boxes_value, masks.shape[0])
    return masks, scores, boxes


In [ ]:
instances = []

for prompt_idx, (label, prompt) in enumerate(flat_prompts, start=1):
    print(f"[{prompt_idx:03d}/{len(flat_prompts):03d}] {label}: {prompt}")
    predictor.handle_request(request={"type": "reset_session", "session_id": session_id})
    response = predictor.handle_request(
        request={
            "type": "add_prompt",
            "session_id": session_id,
            "frame_index": 0,
            "text": prompt,
            "output_prob_thresh": MIN_SCORE,
        }
    )
    masks, scores, boxes = extract_arrays(response["outputs"])
    for i in range(masks.shape[0]):
        score = float(scores[i])
        if score < MIN_SCORE:
            continue
        instances.append(
            {
                "label": label,
                "class_id": int(label_to_id[label]),
                "prompt": prompt,
                "score": score,
                "mask": masks[i].astype(bool),
                "box": None if boxes is None else boxes[i].tolist(),
            }
        )

print("Accepted instances:", len(instances))
print(dict(sorted({label: sum(1 for x in instances if x['label'] == label) for label in label_to_id}.items())))


In [ ]:
semantic_mask = np.zeros((height, width), dtype=np.uint16)
confidence = np.zeros((height, width), dtype=np.float32)

for item in sorted(instances, key=lambda x: x["score"]):
    mask = item["mask"]
    accept = mask & (item["score"] >= confidence)
    semantic_mask[accept] = np.uint16(item["class_id"])
    confidence[accept] = np.float32(item["score"])

class_pixel_counts = {
    label: int(np.count_nonzero(semantic_mask == class_id))
    for label, class_id in label_to_id.items()
    if int(np.count_nonzero(semantic_mask == class_id)) > 0
}
class_pixel_counts


In [ ]:
def color_for_class(class_id: int) -> np.ndarray:
    rng = np.random.default_rng(class_id * 1009 + 17)
    return rng.integers(40, 240, size=3, dtype=np.uint8)


image_np = np.asarray(image).copy()
overlay = image_np.copy()
alpha = 0.45

for class_id in sorted(int(x) for x in np.unique(semantic_mask) if int(x) != 0):
    color = color_for_class(class_id)
    mask = semantic_mask == class_id
    overlay[mask] = (overlay[mask].astype(np.float32) * (1.0 - alpha) + color.astype(np.float32) * alpha).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
axes[0].imshow(image_np)
axes[0].set_title("image")
axes[1].imshow(semantic_mask, interpolation="nearest")
axes[1].set_title("semantic class id")
axes[2].imshow(overlay)
axes[2].set_title("overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout();


In [ ]:
run_dir = OUT_DIR / IMAGE_PATH.stem
run_dir.mkdir(parents=True, exist_ok=True)

np.save(run_dir / "semantic_mask.npy", semantic_mask)
np.save(run_dir / "confidence.npy", confidence)

if instances:
    np.savez_compressed(
        run_dir / "instances.npz",
        masks=np.stack([x["mask"] for x in instances], axis=0).astype(bool),
        scores=np.asarray([x["score"] for x in instances], dtype=np.float32),
        labels=np.asarray([x["label"] for x in instances]),
        class_ids=np.asarray([x["class_id"] for x in instances], dtype=np.uint16),
        prompts=np.asarray([x["prompt"] for x in instances]),
    )
else:
    np.savez_compressed(
        run_dir / "instances.npz",
        masks=np.zeros((0, height, width), dtype=bool),
        scores=np.zeros((0,), dtype=np.float32),
        labels=np.asarray([], dtype=str),
        class_ids=np.zeros((0,), dtype=np.uint16),
        prompts=np.asarray([], dtype=str),
    )

Image.fromarray(overlay).save(run_dir / "overlay.jpg", quality=95)
(run_dir / "metadata.json").write_text(
    json.dumps(
        {
            "image_path": str(IMAGE_PATH),
            "sam3_root": str(SAM3_ROOT),
            "sam3_checkpoint": str(SAM3_CHECKPOINT),
            "prompt_config": str(PROMPT_CONFIG),
            "classes_yaml": str(CLASSES_YAML),
            "min_score": MIN_SCORE,
            "instances": len(instances),
            "class_pixel_counts": class_pixel_counts,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved to", run_dir)


In [ ]:
predictor.handle_request(request={"type": "close_session", "session_id": session_id})
print("Closed SAM3 session")
